# 08 · Cluster stability and clustering settings

**Owner:** Rishi · **Week 3** (support on clustering: test stability, help choose settings) · Code: `src/clustering.py`, tests in `tests/test_clustering.py`

**Questions:**
1. How stable are the communities from one window to the next?
2. Do the cluster features add information beyond the overall correlation level we already have?
3. Which settings give communities that are stable, match sectors, and are not just a copy of market-wide correlation?

**Part A** reads Grace's saved Week 3 results (`notebooks/outputs/grace_week3/`), so it runs anywhere. **Part B** recomputes communities with other settings and needs `data/processed/returns_daily.parquet` and `features/connectedness_60d.parquet` (run `04_returns_and_windows.ipynb` first).

**Sector labels** come from Grace's membership file (one sector per ticker, from the source she documented in `06_clustering_analysis.ipynb`). One label per ticker for all years is an approximation: sector definitions changed in 2016 and 2018.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src import clustering as cl

GRACE = ROOT / "notebooks" / "outputs" / "grace_week3"
PROC = ROOT / "data" / "processed"
episodes = pd.read_csv(ROOT / "docs" / "stress_periods.csv", parse_dates=["start", "end"])

def in_stress(idx):
    s = pd.Series(False, index=idx)
    for _, e in episodes.iterrows():
        s[(idx >= e["start"]) & (idx <= e["end"])] = True
    return s

def shade(ax):
    for _, e in episodes.iterrows():
        ax.axvspan(e["start"], e["end"], color="tab:red", alpha=0.12, lw=0)

## Part A · Grace's Week 3 results (raw correlations, |corr| > 0.55, greedy modularity, every 20 trading days)

In [ ]:
gm = pd.read_csv(GRACE / "community_memberships.csv", parse_dates=["date"])
gf = pd.read_csv(GRACE / "cluster_features.csv", parse_dates=["date"]).set_index("date")
sectors = gm.dropna(subset=["sector"]).drop_duplicates("ticker").set_index("ticker")["sector"].to_dict()

labels_by_date = {d: dict(zip(x["ticker"], x["community"])) for d, x in gm.groupby("date")}
dates = sorted(labels_by_date)
gf["stability"] = pd.Series({b: cl.stability(labels_by_date[a], labels_by_date[b]) for a, b in zip(dates[:-1], dates[1:])})
gf["stress"] = in_stress(gf.index)

print(f"{len(gf)} windows, {len(sectors)} tickers with a sector label")
print("\nStability (ARI vs the window 20 trading days earlier; 1 = same groups, 0 = random):")
display(gf["stability"].describe()[["mean", "25%", "50%", "75%"]].round(3).to_frame("value").T)
print("Medians, calm vs stress windows:")
display(gf.groupby("stress")[["density", "n_communities", "modularity", "largest_community_share", "sector_ari", "stability"]]
        .median().round(3).rename(index={False: "calm", True: "stress"}))
print("Rank correlation of each cluster measure with network density:")
display(gf[["n_communities", "modularity", "largest_community_share", "sector_ari"]]
        .corrwith(gf["density"], method="spearman").round(3).to_frame("spearman vs density"))

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
axes[0].plot(gf.index, gf["density"]); axes[0].set_ylabel("Network density")
axes[1].plot(gf.index, gf["n_communities"], color="tab:purple"); axes[1].set_ylabel("Communities")
axes[2].plot(gf.index, gf["stability"], color="tab:green"); axes[2].set_ylabel("Stability (ARI)")
for ax in axes:
    shade(ax)
axes[0].set_title("Grace's Week 3 clustering: the community count mirrors density (shaded = stress episodes)")
plt.tight_layout(); plt.show()

**Reading Part A:** with a fixed threshold on raw correlations, the network gets denser whenever all stocks move together, so the number of communities and modularity mostly repeat the correlation level (rank correlation close to -1 with density). Agreement with sectors is low and falls further in stress. Groups also change a lot from one window to the next. These are signs that the clusters mostly reflect the market-wide move, not stable groups of stocks.

## Part B · Compare settings on the same windows

Every setting uses the same trailing 60-day windows every 20 trading days (from `src/windows.py`). The baseline row is Grace's saved result; the others are recomputed:

| Setting | Market move removed? | Links kept | Method |
|---|---|---|---|
| Baseline (Grace) | no | \|corr\| > 0.55 | greedy modularity |
| raw, top 5% | no | strongest 5% of positive links | Louvain, weighted |
| market-adjusted, top 5% (recommended) | yes | strongest 5% of positive links | Louvain, weighted |

Scored on: **sector agreement** (higher = groups look like real industries), **stability** (higher = groups persist), and **redundancy** (rank correlation of the community count with average correlation from `connectedness_60d`; close to 0 = new information).

In [ ]:
from src import returns as rt
rets = rt.load_returns()
conn = pd.read_parquet(PROC / "features" / "connectedness_60d.parquet")

settings = [
    cl.Setting("raw, top 5%, Louvain", adjust="none", density=0.05),
    cl.RECOMMENDED,
]
results = {}
for s in settings:
    feats, members = cl.run_over_time(rets, s, window=60, step=20, sectors=sectors)
    results[s.name] = (feats, members)
    print(f"done: {s.name} ({len(feats)} windows)")

def score(feats):
    j = feats.join(conn["avg_corr"], how="left")
    st = in_stress(feats.index)
    return {"sector ARI (median)": feats["sector_ari"].median(),
            "sector ARI, stress": feats.loc[st, "sector_ari"].median(),
            "stability (median)": feats["stability"].median(),
            "communities (median)": feats["n_communities"].median(),
            "redundancy: rank corr of communities with avg corr": j["n_communities"].corr(j["avg_corr"], method="spearman"),
            "redundancy: rank corr of modularity with avg corr": j["modularity"].corr(j["avg_corr"], method="spearman")}

base = gf.rename(columns={"largest_community_share": "largest_share"})
table = pd.DataFrame({"baseline (Grace)": score(base), **{n: score(f) for n, (f, _) in results.items()}}).round(3)
display(table)

In [ ]:
rec = results[cl.RECOMMENDED.name][0]
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for lab, f in [("baseline (Grace)", base), ("market-adjusted, top 5%", rec)]:
    axes[0].plot(f.index, f["sector_ari"], label=lab, lw=0.9)
    axes[1].plot(f.index, f["stability"], label=lab, lw=0.9)
    axes[2].plot(f.index, f["n_communities"], label=lab, lw=0.9)
axes[0].set_ylabel("Sector ARI"); axes[1].set_ylabel("Stability (ARI)"); axes[2].set_ylabel("Communities")
axes[0].legend(loc="upper left")
for ax in axes:
    shade(ax)
axes[0].set_title("Baseline vs market-adjusted clustering (shaded = stress episodes)")
plt.tight_layout(); plt.show()

### Sensitivity of the recommended setting

How much do the scores move with the share of links kept and Louvain's resolution? Run on every 60th trading day to keep it quick.

In [ ]:
rows = []
for dens in [0.02, 0.05, 0.10]:
    for res in [0.5, 1.0, 1.5]:
        s = cl.Setting(f"d={dens}, r={res}", density=dens, resolution=res)
        f, _ = cl.run_over_time(rets, s, window=60, step=60, sectors=sectors)
        rows.append({"density": dens, "resolution": res, "sector ARI": f["sector_ari"].median(),
                     "stability (60 days apart)": f["stability"].median(), "communities": f["n_communities"].median()})
sens = pd.DataFrame(rows).round(3)
display(sens.pivot(index="density", columns="resolution", values="sector ARI").add_prefix("sector ARI, r="))
display(sens.pivot(index="density", columns="resolution", values="communities").add_prefix("communities, r="))

## Save the recommended cluster features

In [ ]:
out = PROC / "features"
out.mkdir(exist_ok=True)
rec_feats, rec_members = results[cl.RECOMMENDED.name]
rec_feats.to_parquet(out / "clusters_market_adjusted_20d.parquet")
rec_members.to_parquet(out / "cluster_memberships_market_adjusted_20d.parquet")
print("Saved features/clusters_market_adjusted_20d.parquet and cluster_memberships_market_adjusted_20d.parquet")

## Notes for Grace

* **To use the recommended setting in your notebook:**
  ```python
  from src import clustering as cl
  feats, members = cl.run_over_time(rets, cl.RECOMMENDED, window=60, step=20, sectors=sector_lookup)
  ```
  `cl.BASELINE` reproduces your current setting for comparison.
* Keep the **network density from the raw-correlation network** as its own feature (it is a good stress measure), but use the market-adjusted communities for questions about *which groups* are under stress.
* Before trusting any setting, check the sensitivity tables: a setting is only useful if the conclusions hold for nearby choices.
* Sector labels: one label per ticker for all years is an approximation (see the note at the top).